# Debug: weights & systematics filling

Diagnoses why the slice plot only shows statistical uncertainty and no
systematic band. This notebook reads ROOT files directly with `uproot`
(no PyROOT/ROOT install needed — just the `.analysis_venv` in this repo),
so it works the same in a plain Jupyter session as it does under `xsnotebook`.

It checks three places where a "systematics disappear" bug typically hides,
in order from raw inputs to the final covariance:

1. **Raw ntuple weight branches** — are the multisim/unisim weight vectors
   actually there and varying, before `univmake` even touches them?
2. **Per-file `univmake` universes** — did `UniverseMaker::build_universes`
   fill each systematic universe histogram with varying (non-CV) content?
3. **The cached `total_*` universes** — this is the one that's easy to miss.
   `SystematicsCalculator` POT-sums all per-file universes into a
   `total_<file_properties_path>` `TDirectoryFile` *once*, then on every
   later run it just reloads that cache if the directory already exists
   (see `SystematicsCalculator::SystematicsCalculator`, which calls
   `load_universes()` instead of `build_universes()` whenever
   `total_subdir` is non-null). If that cache was written by an older,
   buggy run (e.g. before a POT-path fix), every subsequent slice plot
   silently reads the stale numbers — including if they're all zero —
   and nothing downstream will ever complain, because zero MC content just
   means every fractional systematic (`err/y` with `y == 0`) is defined
   away to zero.

Detector-variation and dirt-normalization lines are expected to be absent
here (commented out in `configs/systcalc_numi.conf`), so this notebook
doesn't check for them.

Edit the paths in the next cell to match your setup, then run top to bottom.


In [ ]:
import re
from collections import defaultdict

import uproot
import numpy as np

# ── Configuration — edit these to match your paths ──────────────────────
UNIVMAKE_FILE = "/exp/uboone/data/users/abarnard/analysis/univmake_output/v1_dummy/univmake_double_differential.root"
FPM_CONFIG    = "configs/file_properties.txt"     # path as passed to FilePropertiesManager
SYST_CONFIG   = "configs/systcalc_numi.conf"

# One raw (pre-univmake) MC ntuple file to spot-check, and the TTree name
# used inside it (see configs/systcalc_numi.conf / RunUnivmake.sh)
RAW_MC_FILE = "/exp/uboone/data/users/abarnard/analysis/selection_output/xsec-ana-checkout_MCC9.10_Run4b_v10_04_07_20_NuMI_RHC_nu_overlay_patch_retuple_retuple_hist_bdt_weighted.root"
RAW_TREE_NAME = "XSecAnalyzer/NuMICC1eNp"


## Helper functions

In [ ]:
def top_directory(univmake_file):
    """Return the single top-level TDirectoryFile written by univmake."""
    f = uproot.open(univmake_file)
    top_name = f.keys(recursive=False)[0].split(';')[0]
    return f[top_name], top_name


def list_subdirs(directory):
    """Names of TDirectoryFile children (one per input ntuple file, plus any
    cached 'total_*' directories)."""
    return [k.split(';')[0] for k, c in directory.iterclassnames(recursive=False)
            if c == "TDirectory"]


def universe_groups(directory):
    """Group histogram keys by universe name, e.g.

        {'weight_All_UBGenie': [0, 1, ..., 599], 'unweighted': [0], ...}

    Mirrors the key-parsing logic in SystematicsCalculator::load_universes /
    build_universes (split on the last '_', strip the '_2d' suffix).
    """
    names = [k.split(';')[0] for k, c in directory.iterclassnames(recursive=False)
             if c in ("TH1D", "TH2D")]
    groups = defaultdict(set)
    for n in names:
        m = re.match(r'(.+)_(\d+)_(reco|true|2d|categ|reco2d|true2d)$', n)
        if m:
            groups[m.group(1)].add(int(m.group(2)))
    return {k: sorted(v) for k, v in groups.items()}


def reco_sum(directory, univ_name, idx):
    """Integral of the reco-space histogram for one universe (a cheap proxy
    for 'did this universe get filled at all')."""
    h = directory[f"{univ_name}_{idx}_reco"]
    return float(h.values().sum())


def spread_across_universes(directory, univ_name, indices, sample=8):
    """Reco-integral for a handful of universes, to check for variation
    (not just nonzero content — a bug can leave every universe identical
    to the CV, which is just as fatal for systematics as all-zero)."""
    idxs = indices if len(indices) <= sample else (
        list(indices[:sample // 2]) + list(indices[-(sample - sample // 2):])
    )
    return {i: reco_sum(directory, univ_name, i) for i in idxs}


## 1. Raw ntuple weight branches

Before `univmake` runs at all: are the multisim/unisim weight vectors present
with the expected type/length, and do they actually vary event-to-event
(not just a vector of `1.0`s)?


In [ ]:
raw = uproot.open(RAW_MC_FILE)[RAW_TREE_NAME]
weight_branches = sorted(b for b in raw.keys() if b.startswith("weight_"))
scalar_cv_branches = ["tuned_cv_weight", "ppfx_cv_weight", "normalisation_weight"]

print(f"{RAW_MC_FILE}\n  tree: {RAW_TREE_NAME}  entries: {raw.num_entries}\n")

print(f"{'branch':38s} {'type':20s} {'len(evt0)':>10s} {'mean(evt0)':>12s}")
sample = raw.arrays(weight_branches + scalar_cv_branches, entry_stop=200)
for b in weight_branches:
    v0 = sample[b][0]
    print(f"{b:38s} {raw[b].typename:20s} {len(v0):10d} {np.mean(v0):12.4f}")

print()
for b in scalar_cv_branches:
    vals = sample[b].to_numpy()
    print(f"{b:38s} scalar   mean={vals.mean():.4f}  min={vals.min():.4f}  max={vals.max():.4f}")


Expect: every `weight_*` branch is a non-empty `std::vector<double>`, and the
per-event mean isn't pinned at exactly `1.0000` (that would mean the
reweighting itself never ran upstream). The three scalar CV branches should
vary too — if `tuned_cv_weight`/`ppfx_cv_weight`/`normalisation_weight` were
stuck at `1.0` for every event it would mean `TTree::SetBranchAddress` never
actually bound (wrong branch name/type), silently leaving `UniverseMaker`'s
default value of `1` in place every event.


### Aside: sanity-check the CV correction weights themselves

`UniverseMaker::apply_cv_correction_weights` multiplies every reweightable
universe by `tuned_cv_weight` / `ppfx_cv_weight` / `normalisation_weight`
(NuMI mode), then `safe_weight()` resets anything non-finite or outside
`[0, 30]` back to `1.0` before filling histograms (see
`include/XSecAnalyzer/UniverseMaker.hh`). A small fraction of GENIE-tuned
events landing outside that range is expected and already handled — this
cell just quantifies how many, so a silent `safe_weight()` fallback doesn't
get mistaken for "the weights aren't varying" later on.


In [ ]:
tcw = raw["tuned_cv_weight"].array(entry_stop=200_000).to_numpy()
n = len(tcw)
n_nonfinite = int(np.sum(~np.isfinite(tcw)))
finite = tcw[np.isfinite(tcw)]
n_out_of_range = int(np.sum((finite < 0.) | (finite > 30.)))

print(f"tuned_cv_weight over first {n} events:")
print(f"  non-finite (inf/nan):        {n_nonfinite}  ({100*n_nonfinite/n:.2f}%)")
print(f"  finite but outside [0, 30]:  {n_out_of_range}  ({100*n_out_of_range/n:.2f}%)")
print(f"  finite, in-range mean:       {finite[(finite >= 0.) & (finite <= 30.)].mean():.4f}")
print()
print("Both categories above get reset to 1.0 by safe_weight() before any "
      "histogram is filled, so they dilute a systematic's variance slightly "
      "but do not zero it out.")


## 2. Per-file `univmake` universes

Open the `univmake` output and check, for one input file at a time, that
each systematic's universes were filled and that they actually differ from
each other (not just from zero).


In [ ]:
root, top_name = top_directory(UNIVMAKE_FILE)
subdirs = list_subdirs(root)

mc_subdirs = [s for s in subdirs if not s.startswith("total_")]
total_subdirs = [s for s in subdirs if s.startswith("total_")]

print(f"Top TDirectoryFile: {top_name}\n")
print(f"{len(mc_subdirs)} per-file subdirectories:")
for s in mc_subdirs:
    print(" ", s)
print(f"\n{len(total_subdirs)} cached 'total_*' subdirectories:")
for s in total_subdirs:
    print(" ", s)


In [ ]:
# Pick the subdirectory whose name contains a recognizable fragment of
# RAW_MC_FILE so we're comparing the same sample end-to-end.
import os
frag = os.path.basename(RAW_MC_FILE).split('_')[0:4]
target = next((s for s in mc_subdirs if all(part in s for part in frag[:2])), mc_subdirs[0])
print("Inspecting:", target, "\n")

sd = root[target]
groups = universe_groups(sd)

print(f"{'universe':38s} {'#universes':>10s}")
for name in sorted(groups):
    print(f"{name:38s} {len(groups[name]):10d}")


In [ ]:
# For each multisim/unisim systematic, print the reco-space integral in a
# handful of universes. Flag anything where every universe is identical
# (bitwise) to the first one -- that's the "weights aren't varying" bug.
print(f"{'universe':32s} {'reco sums (sample of universes)'}")
for name in sorted(groups):
    if name in ("unweighted",):
        continue
    idxs = groups[name]
    sums = spread_across_universes(sd, name, idxs)
    vals = list(sums.values())
    flat = len(set(round(v, 6) for v in vals)) == 1 and len(vals) > 1
    flag = "  <-- IDENTICAL ACROSS UNIVERSES" if flat else ""
    print(f"{name:32s} {vals}{flag}")

print()
print("unweighted (CV, no reweighting) reco sum:", reco_sum(sd, 'unweighted', 0))


Expect real spread between universes for anything with >1 universe (GENIE
multisims, flux multisims, reint). A flat line flagged above means the
per-event weight vector for that systematic collapsed to a constant in this
*specific* input file — worth cross-checking against section 1 for that
same file.


## 3. Cached `total_*` universes — the usual suspect

`SystematicsCalculator` POT-sums every per-file universe into a
`total_<mangled fpm config path>` `TDirectoryFile` the first time it runs,
then reuses it forever after (it only checks whether the directory
*exists*, never whether its contents are still valid for the current code/
config/input files). If you've iterated on the selection, the POT lookup,
or `systcalc_numi.conf` while reusing the same `univmake` output file, the
`total_*` directory can quietly keep serving pre-fix numbers — including
zeros — no matter how many times you fix the bug upstream.

This cell repeats the same nonzero/variation check as section 2, but on
every `total_*` directory found, and compares it against the per-file sums.


In [ ]:
if not total_subdirs:
    print("No cached 'total_*' directory found — SystematicsCalculator will "
          "build universes fresh from the per-file histograms on next load.")

for t in total_subdirs:
    tsd = root[t]
    tgroups = universe_groups(tsd)

    print(f"=== {t} ===")
    unw = reco_sum(tsd, "unweighted", 0)
    print(f"  unweighted (CV) reco sum: {unw}")

    for name in ("weight_All_UBGenie", "weight_ppfx_all", "weight_reint_all"):
        if name not in tgroups:
            print(f"  {name}: NOT PRESENT in cache")
            continue
        idxs = tgroups[name]
        sums = spread_across_universes(tsd, name, idxs, sample=4)
        print(f"  {name} sample sums: {list(sums.values())}")

    # Data/EXT are accumulated the same way regardless of the caching path,
    # so they're a useful control: if they're nonzero but the MC is zero,
    # the bug is specific to the MC POT-scaling/accumulation step.
    for data_name in ("onBNB_reco", "extBNB_reco"):
        try:
            v = tsd[data_name].values().sum()
            print(f"  {data_name} sum: {v}")
        except KeyError:
            pass

    verdict = "ALL MC UNIVERSES ARE ZERO -- STALE/BROKEN CACHE" if unw == 0 else "looks populated"
    print(f"  VERDICT: {verdict}\n")


If the verdict above says **STALE/BROKEN CACHE**, that directory is what your
slice-plotting code (`SlicePlots` / `MCC9SystematicsCalculator`) is actually
reading — not the per-file histograms from section 2, no matter how correct
those are. That fully explains "only statistical uncertainty, no systematics
band": the CV prediction and every systematic universe are zero, so every
fractional uncertainty (`err/y`, only computed when `y > 0`) evaluates to
zero and gets silently skipped in the stacked/legend plotting loop in
`Slice_Plots.C`.

### Fix

Because ROOT doesn't reliably reclaim space when you delete a key from a
`TDirectoryFile` opened in `update` mode, the safest fix is to regenerate a
fresh `univmake` output file rather than surgically deleting the `total_*`
key in place:

```bash
./scripts/RunUnivmake.sh v2_after_fixes   # new version tag -> new output file
```

If you'd rather keep the existing file and strip just the stale cache
in-place (ROOT only, since `uproot` can't delete keys), from an `xsroot`
session:

```cpp
TFile f("univmake_double_differential.root", "update");
f.cd("<top TDirectoryFile name printed above>");
gDirectory->Delete("total_configs+file_properties.txt;*");
// delete every other 'total_*' key printed above the same way
f.Close();
```

Either way, re-run this notebook against the new/edited file afterward —
section 3 should then show nonzero, varying sums matching section 2.

Going forward: bump the version tag passed to `RunUnivmake.sh` (or otherwise
regenerate univmake output) any time the selection code, systematics config,
or POT-handling changes, rather than re-running into the same output file —
`SystematicsCalculator` has no way to know the cache is stale.


## 4. Cross-check `systcalc_numi.conf` against the universes that actually exist

Catches typos/renames between the systematics config and the branch names
`univmake` produced — a mismatch here throws `Missing weight key ...` at
`get_covariances()` time, but it's easy to lose that error in a notebook or
batch log.


In [ ]:
def strip_comments(path):
    kept = []
    with open(path) as fh:
        for line in fh:
            s = line.strip()
            if not s or s.startswith('#'):
                continue
            kept.append(line)
    return " ".join(kept).split()


def parse_syst_config(path):
    """Minimal re-implementation of SystematicsCalculator::get_covariances's
    config parsing, just enough to extract every weight_key referenced by an
    RW/FluxRW entry."""
    tok = strip_comments(path)
    i, entries = 0, []
    while i < len(tok):
        name, ctype = tok[i], tok[i + 1]
        i += 2
        if ctype == "sum":
            count = int(tok[i]); i += 1
            i += count
        elif ctype in ("RW", "FluxRW"):
            weight_key = tok[i]; i += 2
            entries.append((name, ctype, weight_key))
        elif ctype == "MCFullCorr":
            i += 1
        elif ctype == "MCFullCorrCategory":
            i += 2
        elif ctype == "DV":
            i += 1
        elif ctype in ("MCstat", "EXTstat", "BNBstat", "AltUniv"):
            pass
        else:
            raise ValueError(f"Unrecognized covariance matrix type '{ctype}' "
                              f"for entry '{name}' -- update this parser if "
                              f"systcalc_numi.conf grew a new type")
    return entries


rw_entries = parse_syst_config(SYST_CONFIG)
available_universe_names = set(groups)  # from section 2, per-file subdirectory

print(f"{'cov. matrix name':24s} {'type':10s} {'weight_key':32s} status")
all_ok = True
for name, ctype, weight_key in rw_entries:
    ok = weight_key in available_universe_names
    all_ok &= ok
    status = "OK" if ok else "MISSING FROM UNIVMAKE OUTPUT"
    print(f"{name:24s} {ctype:10s} {weight_key:32s} {status}")

print()
print("All RW/FluxRW weight keys resolved." if all_ok
      else "Some weight keys are missing -- get_covariances() will throw for these.")


## Summary

- **Section 1** confirms the multisim/unisim weight vectors exist and vary
  in the raw ntuple, before `univmake` runs.
- **Section 2** confirms `UniverseMaker` filled each systematic's universes
  in the `univmake` output with genuinely varying content, per input file.
- **Section 3** checks the POT-summed `total_*` cache that
  `SystematicsCalculator` actually reads at slice-plot time — this is where
  a stale cache from an earlier (pre-fix) run can silently override
  everything upstream being correct.
- **Section 4** guards against a config/branch-name mismatch that would
  otherwise only surface as a runtime exception.

Re-run this notebook after regenerating `univmake` output (or purging the
stale `total_*` cache) to confirm section 3 now agrees with section 2.
